# Sleep dashboard walkthrough
This notebook is a guided demo of the eywa_trees dashboard on the Sleep Health and Lifestyle dataset. It is documentation-style: we load the data once, then show how to explore the tool for both a regression task and a classification task.


In [ ]:
import threading

import pandas as pd

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

from eywa_trees import EywaTreesDash

seed = 7


def start_dashboard(app, port: int):
    thread = threading.Thread(target=lambda: app.run(port=port, debug=False), daemon=True)
    thread.start()
    print(f"Dashboard available at http://127.0.0.1:{port}")
    return thread



## Dataset in one minute
- Sleep Health and Lifestyle (374 people) from Kaggle; place the CSV at `data/Sleep_health_and_lifestyle_dataset.csv`.
- Targets used here: `sleep-duration` (hours, regression) and `sleep-disorder` (None / Insomnia / Sleep Apnea, classification).
- Features kept: demographics, occupation, activity and stress levels, BMI category, heart rate, daily steps, and blood pressure split into systolic/diastolic. Only light cleaning and one-hot encoding are applied.


In [ ]:
def clean_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    cols = df.columns.str.lower()
    cols = cols.str.replace(" ", "-", regex=False)
    cols = cols.str.replace("_", "-", regex=False)
    cols = cols.str.replace("--", "-", regex=False)
    df.columns = cols
    return df


def load_sleep(path: str = "data/Sleep_health_and_lifestyle_dataset.csv") -> pd.DataFrame:
    sleep = pd.read_csv(path, index_col="Person ID")
    sleep["Sleep Disorder"] = sleep["Sleep Disorder"].fillna("None").astype("category")
    sleep["Gender"] = sleep["Gender"].astype("category")
    sleep["Occupation"] = sleep["Occupation"].astype("category")
    sleep["BMI Category"] = sleep["BMI Category"].astype("category")
    sleep["Sleep Duration"] = sleep["Sleep Duration"].astype(float)
    sleep["Quality of Sleep"] = sleep["Quality of Sleep"].astype(int)
    sleep["Physical Activity Level"] = sleep["Physical Activity Level"].astype(int)
    sleep["Stress Level"] = sleep["Stress Level"].astype(int)
    sleep["Heart Rate"] = sleep["Heart Rate"].astype(int)
    sleep["Daily Steps"] = sleep["Daily Steps"].astype(int)
    sleep[["systolic", "diastolic"]] = sleep["Blood Pressure"].str.split("/", expand=True).astype(int)
    sleep = sleep.drop(columns="Blood Pressure")
    return clean_feature_names(sleep)


def encode_features(df: pd.DataFrame, target: str) -> pd.DataFrame:
    X = df.drop(columns=[target])
    cat_cols = X.select_dtypes(include=["category"]).columns
    X = pd.get_dummies(X, columns=cat_cols, prefix_sep="_is_", dtype=int)
    X.columns = (
        X.columns.str.lower()
        .str.replace(" ", "-", regex=False)
        .str.replace("_", "-", regex=False)
        .str.replace("--", "-", regex=False)
    )
    return X


In [ ]:
sleep = load_sleep()
sleep.head()



## Regression: predict nightly hours
We use `sleep-duration` as the target, drop the survey score `quality-of-sleep`, one-hot encode the remaining categoricals, and train a small random forest. The dashboard runs in a background thread so you can keep scrolling.


In [ ]:
reg_target = "sleep-duration"

reg_df = sleep.drop(columns=["quality-of-sleep"])
reg_X = encode_features(reg_df, reg_target)
reg_y = reg_df[reg_target]

reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X, reg_y, test_size=0.25, random_state=seed
)


In [ ]:
reg_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=4,
    random_state=seed,
)
reg_model.fit(reg_X_train, reg_y_train)

reg_dash = EywaTreesDash(
    reg_model,
    reg_X_train,
    reg_X_test,
    reg_y_test,
    feature_names=reg_X.columns,
)
reg_dash.config(show_text=True)

import time
stime = time.time()
reg_thread = start_dashboard(reg_dash, port=8061)

print(f"Regression dashboard started in {time.time() - stime:.2f} seconds.")

## XGBoost

Install with `pip install eywa_trees[xgboost]` to run this section.


In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=60,
    max_depth=6,
    learning_rate=0.1,
    random_state=seed,
)

xgb_model.fit(reg_X_train, reg_y_train)

xgb_dash = EywaTreesDash(
    xgb_model,
    reg_X_train,
    reg_X_test,
    reg_y_test,
    feature_names=reg_X.columns,
)
xgb_dash.config(show_text=True)

xgb_thread = start_dashboard(xgb_dash, port=8062)


### Model tab — Sankey view of a tree

Model tab exposes the forest geometry, one tree at a time. The top slider lets the user select which tree to focus on at any time. The left slider applies depth pruning at the visualization level, automatically updating all values.

Nodes are represented by rectangular elements, while edges represent parent-child relationships. Layout runs left → right (root to leaves). Each split emits two flows: by convention, downward represents samples with feature ≤ threshold, and upward represents samples with feature > threshold. Hover a node to see the feature and threshold. This convention reflects a 90º counter-clockwise rotation of the scikit-learn layout convention (left as feature ≤ threshold, and right as feature > threshold).

Edge widths trace how many training rows flow through that branch (`n_train`); leaves with zero training rows are hidden so thin, unused splits disappear.

Colors encode predicted sleep hours at that node; earlier nodes show the weighted average of their children after pruning.

Top-left metric shows the performance the current tree (with the depth limit applied) would achieve on the test set.


### Predict tab — interactively probe feature values

The Predict tab allows fabricating individual entries and inspecting their pathway through the forest.

Left column exposes multiple sliders representing the features in the dataset. A quantile sampling approach ensures that tick markers respect possible values while remaining evenly spaced. The user can synthesize their own input by changing those sliders. "Return to median" resets all sliders to the median index.

On the right, a standard Go plot draws one tree at a time with a tree selector slider; marks are thinned for large forests, but every tree id remains selectable.

The overlaid prediction text comes from the full random forest on the synthesized row built from the slider settings.

Trees are displaced top → down, respecting sklearn conventions (left flow respects `feature ≤ threshold`, while right flow respects `feature > threshold`).

After synthesizing an input element, its trace inside the actual tree is highlighted.


### Boundary tab — latent landscape

The Boundary tab allow to relate a single synthetical data with its place in the prediction boundary of the model.

Feature sliders on the left mirror the Predict tab. 

The map on the right is a 2D latent projection of the training set (t-SNE), colored by model predictions.

Click anywhere on the map: the crosshair stays at the clicked spot, the nearest training sample is decoded to feature bins, and sliders/prediction update to match. Moving sliders also repositions the crosshair to the closest training profile in the latent space.

### Rules tab — clustered decision rules

Rules tab groups decision rules across the forest into clusters and ranks them by coverage.

The table lists each cluster's representative threshold, predicted value, and coverage stats, plus how many rules were merged.

Select a row to highlight the corresponding cluster on the tree plot.


## Classification: flagging sleep disorders
We switch the target to `sleep-disorder` (None / Insomnia / Sleep Apnea), keep the same preprocessing, and train a random forest classifier to show how the dashboard behaves for classification.


In [ ]:
class_target = "sleep-disorder"

class_df = sleep.copy()
class_X = encode_features(class_df, class_target)
class_y = class_df[class_target]
class_names = sorted(class_y.unique())

class_X_train, class_X_test, class_y_train, class_y_test = train_test_split(
    class_X, class_y, test_size=0.25, random_state=seed, stratify=class_y
)

class_model = RandomForestClassifier(
    n_estimators=80,
    max_depth=6,
    min_samples_leaf=4,
    random_state=seed,
)
class_model.fit(class_X_train, class_y_train)

class_dash = EywaTreesDash(
    class_model,
    class_X_train,
    class_X_test,
    class_y_test,
    feature_names=class_X.columns,
    class_names=class_names,
)
class_dash.config(show_text=True)

class_thread = start_dashboard(class_dash, port=8062)


### How to read the classification dashboard
- **Model tab**: Sankey coloring switches to predicted class; the metric becomes accuracy on the held-out set. Depth and tree sliders work as before, letting you see which trees are most confident about a disorder.
- **Predict tab**: the overlay shows the predicted class label; colors in the Go plot follow the majority class at each node. Use it to probe class flips when toggling lifestyle variables.
- **Rules tab**: rows list clustered rules with coverage stats and predicted values; select a row to highlight the cluster on the tree.
- Class probabilities at internal nodes aggregate leaf probabilities weighted by training counts, so pruning clarifies which early splits already separate disorders cleanly.
